# **TP3** 
# UC06 - Nettoyage et traitement

## Import des bibliothèques et création de la session Spark 

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence_Traitement")
    .master("local[*]")
    .getOrCreate()
)
print("Version Spark : ", spark.version)
spark

Version Spark :  4.3.0


## Chargement des données

In [24]:
base = os.path.expanduser("~/Documents/IPSSI/DataLake Warehouse Lakehouse/TP/J3/data")

def read(path):
    return (spark.read
            .option("header", False)
            .option("inferSchema", True)
            .option("delimiter", "|")
            .csv("file://" + path)
           )

sales_df = read(base + "/raw/data/store_sales/store_sales.dat")
item_df = read(base + "/raw/data/item/item.dat")
df_date = read(base + "/raw/data/date_dim/date_dim.dat")

In [25]:
n_sales_brut = sales_df.count()
n_item_brut = item_df.count()
print("Lignes brutes : store_sales =", n_sales_brut, "| item =", n_item_brut)

Lignes brutes : store_sales = 667579 | item = 17820


## Nettoyer `item`

In [26]:
# Règle « colonne vide » : retirer _c3
item_df = item_df.drop("_c3")

# Règle « format non harmonisé » : supprimer les espaces (trim), puis N/A, Unknown, NO_SUB_CAT, vide -> NULL
colonnes_texte = [c for c, t in item_df.dtypes if t == "string"]
for c in colonnes_texte:
    texte = F.trim(F.col(c))
    item_df = item_df.withColumn(
        c, F.when(texte.isin("N/A", "Unknown", "NO_SUB_CAT", ""), None).otherwise(texte)
    )

# Renommer les colonnes utiles (noms provisoires)
item_df = (item_df
           .withColumnRenamed("_c0", "id_produit")
           .withColumnRenamed("_c5", "prix")
           .withColumnRenamed("_c6", "cout")
           .withColumnRenamed("_c12", "categorie"))

# Règles « doublon » et « identifiant invalide »
item_df = item_df.dropDuplicates()
item_quarantaine = item_df.filter(F.col("id_produit").isNull())
item_silver = item_df.filter(F.col("id_produit").isNotNull())

# Règle « valeur atypique » : on garde, on signale
item_silver = item_silver.withColumn("cout_superieur_prix", F.col("cout") > F.col("prix"))

print("item : silver =", item_silver.count(), "| quarantaine =", item_quarantaine.count())

item : silver = 17820 | quarantaine = 0


### Vérifications

In [27]:
# Vérification du nettoyage de item
placeholders = ["N/A", "Unknown", "NO_SUB_CAT", ""]

def verdict(ok):
    return "OK" if ok else "PROBLÈME"

# 1. La colonne vide _c3 a bien été retirée
print("1. Colonne _c3 retirée        :", verdict("_c3" not in item_silver.columns))

# 2. Les colonnes renommées existent
attendues = ["id_produit", "prix", "cout", "categorie"]
print("2. Colonnes renommées         :", verdict(all(c in item_silver.columns for c in attendues)))

# 3. Plus de valeurs de remplissage ni d'espaces en trop dans les colonnes texte
colonnes_texte = [c for c, t in item_silver.dtypes if t == "string"]
nb_placeholders = sum(item_silver.filter(F.col(c).isin(placeholders)).count() for c in colonnes_texte)
nb_espaces = sum(item_silver.filter(F.col(c) != F.trim(F.col(c))).count() for c in colonnes_texte)
print("3. Valeurs de remplissage     :", verdict(nb_placeholders == 0), "(", nb_placeholders, "restantes )")
print("   Espaces en trop            :", verdict(nb_espaces == 0), "(", nb_espaces, "restants )")

# 4. Aucune ligne perdue : silver + quarantaine = brut
n_silver = item_silver.count()
n_quarantaine = item_quarantaine.count()
print("4. Lignes (brut / silver / quarantaine) :", n_item_brut, "/", n_silver, "/", n_quarantaine,
      verdict(n_silver + n_quarantaine == n_item_brut))

# 5. L'identifiant produit est non nul et unique
n_distincts = item_silver.select("id_produit").distinct().count()
print("5. id_produit unique          :", verdict(n_distincts == n_silver),
      "(", n_distincts, "distincts sur", n_silver, ")")

# 6. L'indicateur a été calculé (on attendait 7 768 d'après l'exploration)
n_flag = item_silver.filter(F.col("cout_superieur_prix")).count()
print("6. cout_superieur_prix = vrai :", n_flag, "(attendu : 7768)", verdict(n_flag == 7768))

# 7. Valeurs manquantes après nettoyage (les valeurs de remplissage sont devenues NULL)
print("\nValeurs manquantes par colonne texte :")
for c in colonnes_texte:
    nb = item_silver.filter(F.col(c).isNull()).count()
    if nb > 0:
        print("  ", c, ":", nb)

item_silver.show(3, truncate=40)

1. Colonne _c3 retirée        : OK
2. Colonnes renommées         : OK
3. Valeurs de remplissage     : OK ( 0 restantes )
   Espaces en trop            : OK ( 0 restants )
4. Lignes (brut / silver / quarantaine) : 17820 / 17820 / 0 OK
5. id_produit unique          : OK ( 17820 distincts sur 17820 )
6. cout_superieur_prix = vrai : 7768 (attendu : 7768) OK

Valeurs manquantes par colonne texte :
   _c4 : 4
   _c8 : 6
   _c10 : 1576
   _c15 : 2502
   _c18 : 1767
   _c19 : 17820
+----------+----------------+----------+----------------------------------------+-----+-----+-------+-----------------------------------+---+---------------------+----+-----------+----+------------------------------------+------+------------+------+----+----+----+----------------------------------------+-------------------+
|id_produit|             _c1|       _c2|                                     _c4| prix| cout|    _c7|                                _c8|_c9|                 _c10|_c11|  categorie|_c13|          

### c_19 a 100% de valeurs manquantes après remplacement, elle peut donc être supprimée

In [29]:
item_silver = item_silver.drop("_c19")

print("1. Colonne _c19 retirée :", "OK" if "_c19" not in item_silver.columns else "PROBLÈME")

1. Colonne _c19 retirée : OK


## Nettoyer `store_sales`

In [30]:
# Renommer les colonnes utiles (noms provisoires)
sales_df = (sales_df
            .withColumnRenamed("_c0", "cle_date")
            .withColumnRenamed("_c2", "id_produit")
            .withColumnRenamed("_c9", "num_ticket")
            .withColumnRenamed("_c10", "quantite")
            .withColumnRenamed("_c13", "prix_vente")
            .withColumnRenamed("_c22", "profit"))

# Règle « doublon » : lignes identiques
sales_df = sales_df.dropDuplicates()

# Règle « clé de date » : ajouter la vraie date depuis date_dim
dates = df_date.select(F.col("_c0").alias("cle_date_ref"), F.col("_c2").alias("date_vente"))
sales_df = (sales_df
            .join(dates, sales_df["cle_date"] == dates["cle_date_ref"], "left")
            .drop("cle_date_ref"))

# Règle « identifiant invalide » : le produit doit exister dans item
produits = item_silver.select(F.col("id_produit").alias("produit_ref"))
sales_df = sales_df.join(produits, sales_df["id_produit"] == produits["produit_ref"], "left")

# Règles « donnée obligatoire », « date » et « quantité » : une ligne invalide va en quarantaine
invalide = (
    F.col("cle_date").isNull()
    | F.col("id_produit").isNull()
    | F.col("num_ticket").isNull()
    | F.col("quantite").isNull()
    | (F.col("quantite") <= 0)
    | F.col("date_vente").isNull()
    | F.col("produit_ref").isNull()
)
sales_quarantaine = sales_df.filter(invalide)
sales_silver = sales_df.filter(~invalide).drop("produit_ref")

# Règle « valeur atypique » : on garde, on signale, et on ajoute une clé technique
sales_silver = (sales_silver
                .withColumn("vente_a_perte", F.col("profit") < 0)
                .withColumn("prix_nul", F.col("prix_vente") == 0)
                .withColumn("ligne_id", F.monotonically_increasing_id()))

print("store_sales : silver =", sales_silver.count(), "| quarantaine =", sales_quarantaine.count())

store_sales : silver = 667579 | quarantaine = 0


### Vérifications

In [31]:
# Vérification du nettoyage de store_sales
def verdict(ok):
    return "OK" if ok else "PROBLÈME"

sales_silver = sales_silver.cache()   # évite de tout recalculer à chaque comptage

# 1. Colonnes renommées et colonnes ajoutées présentes, colonne temporaire retirée
renommees = ["cle_date", "id_produit", "num_ticket", "quantite", "prix_vente", "profit"]
ajoutees = ["date_vente", "vente_a_perte", "prix_nul", "ligne_id"]
print("1. Colonnes renommées         :", verdict(all(c in sales_silver.columns for c in renommees)))
print("   Colonnes ajoutées          :", verdict(all(c in sales_silver.columns for c in ajoutees)))
print("   Colonne temporaire retirée :", verdict("produit_ref" not in sales_silver.columns))

# 2. Aucune ligne perdue ni dupliquée par les jointures : silver + quarantaine = brut
n_silver = sales_silver.count()
n_quarantaine = sales_quarantaine.count()
print("2. Lignes (brut / silver / quarantaine) :", n_sales_brut, "/", n_silver, "/", n_quarantaine,
      verdict(n_silver + n_quarantaine == n_sales_brut))

# 3. Les lignes de silver respectent bien les règles
nb_invalides = sales_silver.filter(
    F.col("date_vente").isNull() | F.col("id_produit").isNull()
    | F.col("num_ticket").isNull() | F.col("quantite").isNull() | (F.col("quantite") <= 0)
).count()
print("3. Lignes invalides dans silver :", verdict(nb_invalides == 0), "(", nb_invalides, ")")

# 4. La clé technique est unique
n_ids = sales_silver.select("ligne_id").distinct().count()
print("4. ligne_id unique            :", verdict(n_ids == n_silver), "(", n_ids, "distincts sur", n_silver, ")")

# 5. Les indicateurs (attendus si la quarantaine est vide : 495 971 et 3 324)
n_perte = sales_silver.filter(F.col("vente_a_perte")).count()
n_prix_nul = sales_silver.filter(F.col("prix_nul")).count()
print("5. vente_a_perte = vrai       :", n_perte, "(attendu : 495971 si quarantaine vide)")
print("   prix_nul = vrai            :", n_prix_nul, "(attendu : 3324 si quarantaine vide)")

# 6. La conversion de la date a fonctionné
print("\n6. Plage des dates de vente :")
sales_silver.agg(F.min("date_vente"), F.max("date_vente")).show()

sales_silver.select("cle_date", "date_vente", "id_produit", "num_ticket", "quantite",
                    "vente_a_perte", "prix_nul").show(5)

1. Colonnes renommées         : OK
   Colonnes ajoutées          : OK
   Colonne temporaire retirée : OK


2. Lignes (brut / silver / quarantaine) : 667579 / 667579 / 0 OK
3. Lignes invalides dans silver : OK ( 0 )


4. ligne_id unique            : OK ( 667579 distincts sur 667579 )
5. vente_a_perte = vrai       : 495971 (attendu : 495971 si quarantaine vide)
   prix_nul = vrai            : 3324 (attendu : 3324 si quarantaine vide)

6. Plage des dates de vente :
+---------------+---------------+
|min(date_vente)|max(date_vente)|
+---------------+---------------+
|     2001-01-01|     2005-12-13|
+---------------+---------------+

+--------+----------+----------+----------+--------+-------------+--------+
|cle_date|date_vente|id_produit|num_ticket|quantite|vente_a_perte|prix_nul|
+--------+----------+----------+----------+--------+-------------+--------+
|   36890|2001-01-01|      3656|         9|      49|         true|   false|
|   36890|2001-01-01|      9526|        12|       9|         true|   false|
|   36890|2001-01-01|     10000|        31|       5|         true|   false|
|   36892|2001-01-03|      8601|       142|       6|         true|   false|
|   36893|2001-01-04|     14551|       167|    

## Ecrire les résultats en Parquet 

In [32]:
def ecrire(df, nom):
    df.write.mode("overwrite").parquet("file://" + base + "/" + nom)

ecrire(sales_silver, "silver/store_sales")
ecrire(sales_quarantaine, "quarantine/store_sales")
ecrire(item_silver, "silver/item")
ecrire(item_quarantaine, "quarantine/item")
print("Écriture terminée")

26/11/19 09:51:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/11/19 09:51:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/11/19 09:51:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/11/19 09:51:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/11/19 09:51:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/11/19 09:51:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/11/19 09:51:01 WARN MemoryManager: Total allocation exceeds 95.

Écriture terminée


## Vérifier l'effet des règles 

In [33]:
n_silver = sales_silver.count()
n_quarantaine = sales_quarantaine.count()
print("store_sales : brut =", n_sales_brut, "| silver =", n_silver, "| quarantaine =", n_quarantaine)
print("silver + quarantaine =", n_silver + n_quarantaine, "(doit être égal au brut, moins les doublons supprimés)")
sales_silver.select("cle_date", "date_vente", "id_produit", "quantite", "vente_a_perte", "prix_nul").show(5)
sales_silver.agg(F.min("date_vente"), F.max("date_vente")).show()

store_sales : brut = 667579 | silver = 667579 | quarantaine = 0
silver + quarantaine = 667579 (doit être égal au brut, moins les doublons supprimés)
+--------+----------+----------+--------+-------------+--------+
|cle_date|date_vente|id_produit|quantite|vente_a_perte|prix_nul|
+--------+----------+----------+--------+-------------+--------+
|   36890|2001-01-01|      3656|      49|         true|   false|
|   36890|2001-01-01|      9526|       9|         true|   false|
|   36890|2001-01-01|     10000|       5|         true|   false|
|   36892|2001-01-03|      8601|       6|         true|   false|
|   36893|2001-01-04|     14551|      65|         true|   false|
+--------+----------+----------+--------+-------------+--------+
only showing top 5 rows
+---------------+---------------+
|min(date_vente)|max(date_vente)|
+---------------+---------------+
|     2001-01-01|     2005-12-13|
+---------------+---------------+

